# 18. Dense Prediction

box 좌표계와 IoU, mask의 IoU·Dice를 계산하고 배경 우세와 오류 전파를 확인합니다.

핵심 실습은 NumPy와 Python 표준 라이브러리만 사용합니다. API 호출, 모델 다운로드, GPU는 사용하지 않습니다. 수치는 고정된 작은 입력의 계산이며 실제 pretrained model의 성능이 아닙니다.

In [1]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(27)

## Box geometry

좌표는 `(x1, y1, x2, y2)`이며 오른쪽·아래 경계는 제외합니다. 면적에 `+1`을 더하는 구현과 섞지 않습니다.

In [2]:
def box_iou(a,b):
    a,b=np.asarray(a,float),np.asarray(b,float)
    if np.any(a[2:]<a[:2]) or np.any(b[2:]<b[:2]):
        raise ValueError("invalid xyxy box")
    inter=np.maximum(0, np.minimum(a[2:],b[2:])-np.maximum(a[:2],b[:2])).prod()
    area_a=np.maximum(0,a[2:]-a[:2]).prod()
    area_b=np.maximum(0,b[2:]-b[:2]).prod()
    union=area_a+area_b-inter
    return inter/union if union else 1.0
a=[0,0,80,60]
assert box_iou(a,a)==1
assert np.isclose(box_iou(a,[40,0,120,60]),1/3)
assert box_iou(a,[100,0,180,60])==0
print("shift 0,40,100:", [box_iou(a,[s,0,s+80,60])for s in [0,40,100]])

shift 0,40,100: [np.float64(1.0), np.float64(0.3333333333333333), np.float64(0.0)]


## Resize와 crop 되돌리기

원본에서 `(100, 50)`부터 잘라낸 crop을 절반 크기로 줄여 모델에 넣었다고 가정합니다.

In [3]:
box_in_model=np.array([10.,20.,40.,60.])
box_in_original=box_in_model*2 + np.array([100.,50.,100.,50.])
print(box_in_original)
assert np.array_equal(box_in_original, [120,90,180,170])

[120.  90. 180. 170.]


## 작은 객체의 mask 평가

배경만 내는 모델의 높은 pixel accuracy가 객체 성공을 의미하는지 비교합니다.

In [4]:
target=np.zeros((10,10),bool)
target[2:4,3:5]=True
background_only=np.zeros_like(target)
partial=np.zeros_like(target);partial[2:4,4:6]=True
def mask_scores(pred,target):
    intersection=np.logical_and(pred,target).sum()
    union=np.logical_or(pred,target).sum()
    denom=pred.sum()+target.sum()
    return {"accuracy": float((pred==target).mean()),
            "iou": float(intersection/union) if union else 1.,
            "dice": float(2*intersection/denom) if denom else 1.}
print("background only:", mask_scores(background_only,target))
print("partial overlap:", mask_scores(partial,target))
assert mask_scores(background_only,target)["accuracy"]==.96
assert mask_scores(background_only,target)["iou"]==0

background only: {'accuracy': 0.96, 'iou': 0.0, 'dice': 0.0}
partial overlap: {'accuracy': 0.96, 'iou': 0.3333333333333333, 'dice': 0.5}


## 과제

- IoU가 0.5 이상인 shift 범위를 0~80 pixel에서 찾으세요.
- 원본 이미지의 pixel 면적이 0.25 mm²일 때 객체 면적을 구하고, 2배 resize한 mask의 pixel 수를 그대로 쓰면 어떤 오류가 나는지 보이세요.

In [5]:
# 과제 풀이 1
valid=[s for s in range(81)if box_iou(a,[s,0,s+80,60])>=.5]
print("maximum integer shift:", max(valid))
assert max(valid)==26
# 과제 풀이 2
area_mm2=target.sum()*.25
resized=np.repeat(np.repeat(target,2,axis=0),2,axis=1)
wrong=resized.sum()*.25
correct=resized.sum()*(.25/4)
print("original, wrong, corrected:",area_mm2,wrong,correct)
assert np.isclose(correct,area_mm2) and wrong==4*area_mm2

maximum integer shift: 26
original, wrong, corrected: 1.0 4.0 1.0


## 해석

이 실습은 검출·분할 모델 추론을 대신하지 않습니다. 모델 출력의 좌표·면적·지표를 독립 검증하는 코드입니다. 실제 pipeline에서는 원본 위에 box와 mask를 겹쳐 확인합니다. [Grounding DINO](https://arxiv.org/abs/2303.05499), [SAM](https://arxiv.org/abs/2304.02643).

## Full Model Lab
The lecture HTML contains an inline numerical demo and a Model Lab panel. These notebooks run the corresponding pretrained models in a separate Python environment.
- [18_vision-II.ipynb](model-labs/18_vision-II.ipynb)
